# Score fusion

In [1]:
import sys
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Union

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path("/home/hungdx/code/Lightning-hydra").resolve()
REPO_ROOT_STR = "/home/hungdx/code/Lightning-hydra"  # Original path string for replacements


import eval_metrics_DF as em  # noqa: E402

IGNORED_PREFIXES = ("merged_protocol", "merged_scores", "pooled_", "summary")
Reducer = Union[str, Callable[[np.ndarray], float]]


def _resolve_repo_path(path_like: Union[str, Path]) -> Path:
    # Convert to string first to handle Path objects and strings uniformly
    path_str = str(path_like)
    
    # Normalize paths that contain old base paths
    # Handle both /nvme2/hungdx/Lightning-hydra and /nvme2/hungdx/code/Lightning-hydra
    if "/nvme2/hungdx/Lightning-hydra" in path_str and "/nvme2/hungdx/code/Lightning-hydra" not in path_str:
        # Replace /nvme2/hungdx/Lightning-hydra with current REPO_ROOT_STR
        path_str = path_str.replace("/nvme2/hungdx/Lightning-hydra", REPO_ROOT_STR)
    elif "/nvme2/hungdx/code/Lightning-hydra" in path_str:
        # Also normalize /nvme2/hungdx/code/Lightning-hydra to REPO_ROOT_STR
        path_str = path_str.replace("/nvme2/hungdx/code/Lightning-hydra", REPO_ROOT_STR)
    
    path = Path(path_str)
    if path.is_absolute():
        return path.resolve()
    return (REPO_ROOT / path).resolve()


def _read_protocol(protocol_path: Path) -> pd.DataFrame:
    # Normalize the path in case it still contains old base paths
    protocol_path = _resolve_repo_path(protocol_path)
    if not protocol_path.exists():
        raise FileNotFoundError(f"Protocol file not found: {protocol_path}")
    df = pd.read_csv(
        protocol_path,
        sep=r"\s+",
        header=None,
        names=["path", "subset", "label"],
        dtype={"path": str, "subset": str, "label": str},
    )
    # Only use subset is eval
    df = df[df["subset"] == "eval"]
    return df[["path", "label"]]


def _read_score_file(score_path: Path) -> pd.DataFrame:
    if not score_path.exists():
        raise FileNotFoundError(f"Score file not found: {score_path}")
    df = pd.read_csv(
        score_path,
        sep=r"\s+",
        header=None,
        names=["path", "spoof_score", "bona_score"],
        dtype={"path": str, "spoof_score": np.float32, "bona_score": np.float32},
    )
    return df.drop_duplicates(subset="path")


def _discover_datasets(folder: Path) -> Dict[str, Dict[str, Path]]:
    """Return {dataset_name: {"score": Path, "protocol": Path}} for a folder."""
    dataset_map: Dict[str, Dict[str, Path]] = {}
    pooled_meta = sorted(folder.glob("pooled_merged_protocol*.txt"))

    if pooled_meta:
        meta_path = pooled_meta[0]
        with meta_path.open() as handle:
            for line in handle:
                line = line.strip()
                if (
                    not line
                    or line.startswith("#")
                    or line.startswith("TOTAL")
                    or "|" not in line
                ):
                    continue
                parts = [part.strip() for part in line.split("|")]
                if len(parts) < 4:
                    continue
                dataset_name, _, protocol_str, score_str = parts[:4]
                protocol_path = _resolve_repo_path(protocol_str)
                score_path = _resolve_repo_path(score_str)
                if not score_path.exists():
                    fallback = folder / Path(score_str).name
                    if fallback.exists():
                        score_path = fallback
                dataset_map[dataset_name] = {
                    "protocol": protocol_path,
                    "score": score_path,
                }
        return dataset_map

    # Fallback: infer from filenames (best effort)
    for txt_path in folder.glob("*.txt"):
        if txt_path.name.startswith(IGNORED_PREFIXES):
            continue
        dataset_name = txt_path.stem.split("_", 1)[0]
        protocol_candidate = _resolve_repo_path(
            f"data/WildSpoof_Final_Eval/{dataset_name}_bonafide/protocol.txt"
        )
        dataset_map[dataset_name] = {
            "protocol": protocol_candidate,
            "score": txt_path,
        }
    return dataset_map


def _select_columns(prefix: str, columns: Iterable[str]) -> List[str]:
    return [col for col in columns if col.startswith(prefix)]


def _apply_reducer(frame: pd.DataFrame, cols: List[str], reducer: Reducer) -> pd.Series:
    if not cols:
        raise ValueError("No columns provided for aggregation")

    if callable(reducer):
        return frame[cols].apply(lambda row: reducer(row.values.astype(float)), axis=1)

    reducer_key = reducer.lower()
    if reducer_key == "mean":
        return frame[cols].mean(axis=1)
    if reducer_key == "median":
        return frame[cols].median(axis=1)
    if reducer_key == "max":
        return frame[cols].max(axis=1)
    if reducer_key == "min":
        return frame[cols].min(axis=1)
    if reducer_key == "sum":
        return frame[cols].sum(axis=1)

    raise ValueError(f"Unsupported reducer: {reducer}")


def _evaluate_scores(scored_df: pd.DataFrame) -> Dict[str, Any]:
    preds = np.where(scored_df["spoof_score"] < scored_df["bona_score"], "bonafide", "spoof")
    accuracy = float((preds == scored_df["label"].to_numpy()).mean() * 100)

    bona_scores = scored_df.loc[scored_df["label"] == "bonafide", "bona_score"].to_numpy()
    spoof_scores = scored_df.loc[scored_df["label"] == "spoof", "bona_score"].to_numpy()
    eer, threshold = em.compute_eer(bona_scores, spoof_scores)

    return {
        "eer": float(eer * 100),
        "threshold": float(threshold),
        "accuracy": accuracy,
        "min_score": float(scored_df["bona_score"].min()),
        "max_score": float(scored_df["bona_score"].max()),
        "samples": int(len(scored_df)),
        "bonafide_samples": int((scored_df["label"] == "bonafide").sum()),
        "spoof_samples": int((scored_df["label"] == "spoof").sum()),
    }


def fuse_model_score_folders(
    folder_paths: Sequence[Union[str, Path]],
    reducer: Reducer = "mean",
    output_dir: Optional[Union[str, Path]] = None,
) -> Dict[str, Any]:
    """Fuse score files from multiple benchmark folders and compute metrics.

    Args:
        folder_paths: Iterable of directories produced by benchmark runs.
        reducer: Aggregation strategy. Either "mean" (default), "median", "max",
            "min", or a callable that accepts a 1D np.ndarray of scores.
        output_dir: Optional directory to store fused score files (per dataset).

    Returns:
        Dictionary containing fused DataFrames, dataset metrics, pooled metrics,
        and a summary DataFrame for quick inspection.
    """

    model_folders = [Path(p).resolve() for p in folder_paths]
    for folder in model_folders:
        if not folder.exists():
            raise FileNotFoundError(f"Folder does not exist: {folder}")

    catalogs = [_discover_datasets(folder) for folder in model_folders]
    if not catalogs:
        raise RuntimeError("No folders provided for fusion")

    dataset_sets = [set(cat.keys()) for cat in catalogs if cat]
    if not dataset_sets:
        raise RuntimeError("Unable to discover datasets for the provided folders")

    common_datasets = set.intersection(*dataset_sets)
    if not common_datasets:
        raise ValueError("No common datasets across the provided folders")

    protocol_map: Dict[str, Path] = {}
    dataset_score_paths: Dict[str, List[Path]] = {}

    for dataset in common_datasets:
        score_paths: List[Path] = []
        for catalog in catalogs:
            if dataset not in catalog:
                raise ValueError(f"Dataset '{dataset}' missing in one of the folders")
            meta = catalog[dataset]
            protocol_path = _resolve_repo_path(meta["protocol"])  # Ensure normalization
            if dataset not in protocol_map:
                protocol_map[dataset] = protocol_path
            else:
                if protocol_map[dataset].resolve() != protocol_path.resolve():
                    raise ValueError(
                        f"Protocol mismatch for dataset '{dataset}':\n"
                        f"{protocol_map[dataset]} != {protocol_path}"
                    )
            score_paths.append(meta["score"])
        dataset_score_paths[dataset] = score_paths

    output_dir_path: Optional[Path] = None
    if output_dir is not None:
        output_dir_path = Path(output_dir).resolve()
        output_dir_path.mkdir(parents=True, exist_ok=True)

    fused_scores: Dict[str, pd.DataFrame] = {}
    fused_outputs: Dict[str, Optional[Path]] = {}
    dataset_metrics: Dict[str, Dict[str, Any]] = {}
    pooled_entries: List[pd.DataFrame] = []
    summary_rows: List[Dict[str, Any]] = []

    for dataset in sorted(common_datasets):
        merged_df: Optional[pd.DataFrame] = None
        for idx, score_path in enumerate(dataset_score_paths[dataset]):
            df = _read_score_file(score_path).rename(
                columns={
                    "spoof_score": f"spoof_{idx}",
                    "bona_score": f"bona_{idx}",
                }
            )
            merged_df = df if merged_df is None else merged_df.merge(df, on="path", how="inner")

        if merged_df is None:
            raise RuntimeError(f"No scores loaded for dataset '{dataset}'")

        spoof_cols = _select_columns("spoof_", merged_df.columns)
        bona_cols = _select_columns("bona_", merged_df.columns)

        fused_df = pd.DataFrame({"path": merged_df["path"]})
        fused_df["spoof_score"] = _apply_reducer(merged_df, spoof_cols, reducer)
        fused_df["bona_score"] = _apply_reducer(merged_df, bona_cols, reducer)
        fused_scores[dataset] = fused_df

        out_path = None
        if output_dir_path is not None:
            safe_name = dataset.replace(" ", "_")
            out_path = output_dir_path / f"{safe_name}_fusion.txt"
            fused_df.to_csv(
                out_path,
                sep=" ",
                header=False,
                index=False,
                float_format="%.6f",
            )
        fused_outputs[dataset] = out_path

        protocol_df = _read_protocol(protocol_map[dataset])
        scored_df = protocol_df.merge(fused_df, on="path", how="inner")
        missing = len(protocol_df) - len(scored_df)
        if missing > 0:
            print(
                f"⚠️ Dataset '{dataset}': {missing} entries dropped during fusion due to missing paths",
                flush=True,
            )

        metrics = _evaluate_scores(scored_df)
        metrics.update({
            "dataset": dataset,
            "missing_samples": int(missing),
        })

        dataset_metrics[dataset] = metrics
        summary_rows.append(metrics)
        pooled_entries.append(scored_df.assign(dataset=dataset))

    if not pooled_entries:
        raise RuntimeError("No scores available for pooled evaluation")

    pooled_df = pd.concat(pooled_entries, ignore_index=True)
    pooled_metrics = _evaluate_scores(pooled_df)
    pooled_metrics.update({
        "dataset": "POOLED",
        "datasets": len(common_datasets),
        "dataset_counts": {row["dataset"]: row["samples"] for row in summary_rows},
    })

    summary_df = pd.DataFrame(summary_rows).set_index("dataset").sort_index()

    return {
        "fused_scores": fused_scores,
        "fused_paths": fused_outputs,
        "dataset_metrics": dataset_metrics,
        "pooled_metrics": pooled_metrics,
        "summary": summary_df,
    }


In [2]:
example_folders = [
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval_analyze/xlsr_conformertcm_mdt_rawboost3_spoofceleb", # 8.56
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval_analyze/XLSR_ConformerTCM_spoofceleb_clean_ft_new_noise_and_vocoded_Nov16",  # 9.02
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval_analyze/XLSR_ConformerTCM_spoofceleb_clean_ft_new_noise_balacne_ce_Nov16", # 8.17
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval_analyze/XLSR_ConformerTCM_spoofceleb_noise", # 8.39
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval_analyze/XLSR_ConformerTCM_spoofceleb_noise_ft" # 10.53
]

# Uncomment to run the fusion pipeline end-to-end. This will load several large
# files, so expect it to take a few minutes.
fusion_result = fuse_model_score_folders(example_folders, reducer="mean")

# Show fusion EER and supporting metrics per dataset
summary_columns = ["eer", "threshold", "accuracy", "samples", "missing_samples"]
dataset_summary = (
    fusion_result["summary"][summary_columns]
    .sort_index()
    .assign(eer=lambda df: df["eer"].round(4), accuracy=lambda df: df["accuracy"].round(4))
)
display(dataset_summary)

# Pooled metrics across all datasets
fusion_result["pooled_metrics"]  # pooled EER/accuracy overview


/nvme2/hungdx/code/Lightning-hydra/notebooks/WildSpoof/eval_metrics_DF.py:36: RuntimeWarning: invalid value encountered in divide
  far = np.concatenate((np.atleast_1d(1), nontarget_trial_sums / nontarget_scores.size))  # false acceptance rates
/nvme2/hungdx/code/Lightning-hydra/notebooks/WildSpoof/eval_metrics_DF.py:36: RuntimeWarning: invalid value encountered in divide
  far = np.concatenate((np.atleast_1d(1), nontarget_trial_sums / nontarget_scores.size))  # false acceptance rates


,eer,threshold,accuracy,samples,missing_samples
dataset,,,,,
Enroll,NaN,-2.866199,99.7646,10619,0


{'eer': nan,
 'threshold': -2.866199016571045,
 'accuracy': 99.76457293530464,
 'min_score': -2.866199016571045,
 'max_score': 4.381802082061768,
 'samples': 10619,
 'bonafide_samples': 10619,
 'spoof_samples': 0,
 'dataset': 'POOLED',
 'datasets': 1,
 'dataset_counts': {'Enroll': 10619}}

# Merge final protocol and analyze

In [1]:
import sys
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Union

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path("/home/hungdx/code/Lightning-hydra").resolve()
REPO_ROOT_STR = "/home/hungdx/code/Lightning-hydra"  # Original path string for replacements


import eval_metrics_DF as em  # noqa: E402

IGNORED_PREFIXES = ("merged_protocol", "merged_scores", "pooled_", "summary")
Reducer = Union[str, Callable[[np.ndarray], float]]


def _resolve_repo_path(path_like: Union[str, Path]) -> Path:
    # Convert to string first to handle Path objects and strings uniformly
    path_str = str(path_like)
    
    # Normalize paths that contain old base paths
    # Handle both /nvme2/hungdx/Lightning-hydra and /nvme2/hungdx/code/Lightning-hydra
    if "/nvme2/hungdx/Lightning-hydra" in path_str and "/nvme2/hungdx/code/Lightning-hydra" not in path_str:
        # Replace /nvme2/hungdx/Lightning-hydra with current REPO_ROOT_STR
        path_str = path_str.replace("/nvme2/hungdx/Lightning-hydra", REPO_ROOT_STR)
    elif "/nvme2/hungdx/code/Lightning-hydra" in path_str:
        # Also normalize /nvme2/hungdx/code/Lightning-hydra to REPO_ROOT_STR
        path_str = path_str.replace("/nvme2/hungdx/code/Lightning-hydra", REPO_ROOT_STR)
    
    path = Path(path_str)
    if path.is_absolute():
        return path.resolve()
    return (REPO_ROOT / path).resolve()


def _read_protocol(protocol_path: Path) -> pd.DataFrame:
    # Normalize the path in case it still contains old base paths
    protocol_path = _resolve_repo_path(protocol_path)
    if not protocol_path.exists():
        raise FileNotFoundError(f"Protocol file not found: {protocol_path}")
    df = pd.read_csv(
        protocol_path,
        sep=r"\s+",
        header=None,
        names=["path", "subset", "label"],
        dtype={"path": str, "subset": str, "label": str},
    )
    # Only use subset is eval
    df = df[df["subset"] == "eval"]
    return df[["path", "label"]]


def _read_score_file(score_path: Path) -> pd.DataFrame:
    if not score_path.exists():
        raise FileNotFoundError(f"Score file not found: {score_path}")
    df = pd.read_csv(
        score_path,
        sep=r"\s+",
        header=None,
        names=["path", "spoof_score", "bona_score"],
        dtype={"path": str, "spoof_score": np.float32, "bona_score": np.float32},
    )
    return df.drop_duplicates(subset="path")


def _discover_datasets(folder: Path) -> Dict[str, Dict[str, Path]]:
    """Return {dataset_name: {"score": Path, "protocol": Path}} for a folder."""
    dataset_map: Dict[str, Dict[str, Path]] = {}
    pooled_meta = sorted(folder.glob("pooled_merged_protocol*.txt"))

    if pooled_meta:
        meta_path = pooled_meta[0]
        with meta_path.open() as handle:
            for line in handle:
                line = line.strip()
                if (
                    not line
                    or line.startswith("#")
                    or line.startswith("TOTAL")
                    or "|" not in line
                ):
                    continue
                parts = [part.strip() for part in line.split("|")]
                if len(parts) < 4:
                    continue
                dataset_name, _, protocol_str, score_str = parts[:4]
                protocol_path = _resolve_repo_path(protocol_str)
                score_path = _resolve_repo_path(score_str)
                if not score_path.exists():
                    fallback = folder / Path(score_str).name
                    if fallback.exists():
                        score_path = fallback
                dataset_map[dataset_name] = {
                    "protocol": protocol_path,
                    "score": score_path,
                }
        return dataset_map

    # Fallback: infer from filenames (best effort)
    for txt_path in folder.glob("*.txt"):
        if txt_path.name.startswith(IGNORED_PREFIXES):
            continue
        dataset_name = txt_path.stem.split("_", 1)[0]
        protocol_candidate = _resolve_repo_path(
            f"data/WildSpoof_Final_Eval/{dataset_name}/protocol.txt"
        )
        dataset_map[dataset_name] = {
            "protocol": protocol_candidate,
            "score": txt_path,
        }
    return dataset_map


def _select_columns(prefix: str, columns: Iterable[str]) -> List[str]:
    return [col for col in columns if col.startswith(prefix)]


def _apply_reducer(frame: pd.DataFrame, cols: List[str], reducer: Reducer) -> pd.Series:
    if not cols:
        raise ValueError("No columns provided for aggregation")

    if callable(reducer):
        return frame[cols].apply(lambda row: reducer(row.values.astype(float)), axis=1)

    reducer_key = reducer.lower()
    if reducer_key == "mean":
        return frame[cols].mean(axis=1)
    if reducer_key == "median":
        return frame[cols].median(axis=1)
    if reducer_key == "max":
        return frame[cols].max(axis=1)
    if reducer_key == "min":
        return frame[cols].min(axis=1)
    if reducer_key == "sum":
        return frame[cols].sum(axis=1)

    raise ValueError(f"Unsupported reducer: {reducer}")


def _evaluate_scores(scored_df: pd.DataFrame) -> Dict[str, Any]:
    preds = np.where(scored_df["spoof_score"] < scored_df["bona_score"], "bonafide", "spoof")
    accuracy = float((preds == scored_df["label"].to_numpy()).mean() * 100)

    bona_scores = scored_df.loc[scored_df["label"] == "bonafide", "bona_score"].to_numpy()
    spoof_scores = scored_df.loc[scored_df["label"] == "spoof", "bona_score"].to_numpy()
    eer, threshold = em.compute_eer(bona_scores, spoof_scores)

    return {
        "eer": float(eer * 100),
        "threshold": float(threshold),
        "accuracy": accuracy,
        "min_score": float(scored_df["bona_score"].min()),
        "max_score": float(scored_df["bona_score"].max()),
        "samples": int(len(scored_df)),
        "bonafide_samples": int((scored_df["label"] == "bonafide").sum()),
        "spoof_samples": int((scored_df["label"] == "spoof").sum()),
    }


def fuse_model_score_folders(
    folder_paths: Sequence[Union[str, Path]],
    reducer: Reducer = "mean",
    output_dir: Optional[Union[str, Path]] = None,
) -> Dict[str, Any]:
    """Fuse score files from multiple benchmark folders and compute metrics.

    Args:
        folder_paths: Iterable of directories produced by benchmark runs.
        reducer: Aggregation strategy. Either "mean" (default), "median", "max",
            "min", or a callable that accepts a 1D np.ndarray of scores.
        output_dir: Optional directory to store fused score files (per dataset).

    Returns:
        Dictionary containing fused DataFrames, dataset metrics, pooled metrics,
        and a summary DataFrame for quick inspection.
    """

    model_folders = [Path(p).resolve() for p in folder_paths]
    for folder in model_folders:
        if not folder.exists():
            raise FileNotFoundError(f"Folder does not exist: {folder}")

    catalogs = [_discover_datasets(folder) for folder in model_folders]
    print(f"catalogs {catalogs}")
    if not catalogs:
        raise RuntimeError("No folders provided for fusion")

    dataset_sets = [set(cat.keys()) for cat in catalogs if cat]
    if not dataset_sets:
        raise RuntimeError("Unable to discover datasets for the provided folders")

    common_datasets = set.intersection(*dataset_sets)
    if not common_datasets:
        raise ValueError("No common datasets across the provided folders")

    protocol_map: Dict[str, Path] = {}
    dataset_score_paths: Dict[str, List[Path]] = {}

    for dataset in common_datasets:
        score_paths: List[Path] = []
        for catalog in catalogs:
            if dataset not in catalog:
                #raise ValueError(f"Dataset '{dataset}' missing in one of the folders")
                print(f"Dataset '{dataset}' not in the {catalog}")
                continue
            meta = catalog[dataset]
            protocol_path = _resolve_repo_path(meta["protocol"])  # Ensure normalization
            if dataset not in protocol_map:
                protocol_map[dataset] = protocol_path
            else:
                if protocol_map[dataset].resolve() != protocol_path.resolve():
                    raise ValueError(
                        f"Protocol mismatch for dataset '{dataset}':\n"
                        f"{protocol_map[dataset]} != {protocol_path}"
                    )
            score_paths.append(meta["score"])
        dataset_score_paths[dataset] = score_paths

    output_dir_path: Optional[Path] = None
    if output_dir is not None:
        output_dir_path = Path(output_dir).resolve()
        output_dir_path.mkdir(parents=True, exist_ok=True)

    fused_scores: Dict[str, pd.DataFrame] = {}
    fused_outputs: Dict[str, Optional[Path]] = {}
    dataset_metrics: Dict[str, Dict[str, Any]] = {}
    pooled_entries: List[pd.DataFrame] = []
    summary_rows: List[Dict[str, Any]] = []

    for dataset in sorted(common_datasets):
        merged_df: Optional[pd.DataFrame] = None
        for idx, score_path in enumerate(dataset_score_paths[dataset]):
            df = _read_score_file(score_path).rename(
                columns={
                    "spoof_score": f"spoof_{idx}",
                    "bona_score": f"bona_{idx}",
                }
            )
            merged_df = df if merged_df is None else merged_df.merge(df, on="path", how="inner")

        if merged_df is None:
            raise RuntimeError(f"No scores loaded for dataset '{dataset}'")

        spoof_cols = _select_columns("spoof_", merged_df.columns)
        bona_cols = _select_columns("bona_", merged_df.columns)

        fused_df = pd.DataFrame({"path": merged_df["path"]})
        fused_df["spoof_score"] = _apply_reducer(merged_df, spoof_cols, reducer)
        fused_df["bona_score"] = _apply_reducer(merged_df, bona_cols, reducer)
        fused_scores[dataset] = fused_df

        out_path = None
        if output_dir_path is not None:
            safe_name = dataset.replace(" ", "_")
            out_path = output_dir_path / f"{safe_name}_fusion.txt"
            fused_df.to_csv(
                out_path,
                sep=" ",
                header=False,
                index=False,
                float_format="%.6f",
            )
        fused_outputs[dataset] = out_path

        protocol_df = _read_protocol(protocol_map[dataset])
        scored_df = protocol_df.merge(fused_df, on="path", how="inner")
        missing = len(protocol_df) - len(scored_df)
        if missing > 0:
            print(
                f"⚠️ Dataset '{dataset}': {missing} entries dropped during fusion due to missing paths",
                flush=True,
            )

        metrics = _evaluate_scores(scored_df)
        metrics.update({
            "dataset": dataset,
            "missing_samples": int(missing),
        })

        dataset_metrics[dataset] = metrics
        summary_rows.append(metrics)
        pooled_entries.append(scored_df.assign(dataset=dataset))

    if not pooled_entries:
        raise RuntimeError("No scores available for pooled evaluation")

    pooled_df = pd.concat(pooled_entries, ignore_index=True)
    pooled_metrics = _evaluate_scores(pooled_df)
    pooled_metrics.update({
        "dataset": "POOLED",
        "datasets": len(common_datasets),
        "dataset_counts": {row["dataset"]: row["samples"] for row in summary_rows},
    })

    summary_df = pd.DataFrame(summary_rows).set_index("dataset").sort_index()

    return {
        "fused_scores": fused_scores,
        "fused_paths": fused_outputs,
        "dataset_metrics": dataset_metrics,
        "pooled_metrics": pooled_metrics,
        "summary": summary_df,
    }


example_folders = [
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/xlsr_conformertcm_mdt_rawboost3_spoofceleb", # 8.56
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/XLSR_ConformerTCM_spoofceleb_clean_ft_new_noise_and_vocoded_Nov16",  # 9.02
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/XLSR_ConformerTCM_spoofceleb_clean_ft_new_noise_balacne_ce_Nov16", # 8.17
    #"/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/XLSR_ConformerTCM_spoofceleb_noise", # 8.39
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/XLSR_ConformerTCM_spoofceleb_noise_ft", # 10.53
    
    #"/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/XLSR_ConformerTCM_LoRA_lossy_codec-conf-1"
    "/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/XLSR_ConformerTCM_LoRA_lossy_codec-conf-1-v2"
]

# Uncomment to run the fusion pipeline end-to-end. This will load several large
# files, so expect it to take a few minutes.
fusion_result = fuse_model_score_folders(example_folders, reducer="mean")

# Show fusion EER and supporting metrics per dataset
summary_columns = ["eer", "threshold", "accuracy", "samples", "missing_samples"]
dataset_summary = (
    fusion_result["summary"][summary_columns]
    .sort_index()
    .assign(eer=lambda df: df["eer"].round(4), accuracy=lambda df: df["accuracy"].round(4))
)
display(dataset_summary)

# Pooled metrics across all datasets
fusion_result["pooled_metrics"]  # pooled EER/accuracy overview


catalogs [{'Enroll_bonafide': {'protocol': PosixPath('/nvme2/hungdx/code/Lightning-hydra/data/WildSpoof_Final_Eval/Enroll_bonafide/protocol.txt'), 'score': PosixPath('/nvme2/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/xlsr_conformertcm_mdt_rawboost3_spoofceleb/Enroll_bonafide_wildspoof_Nov3_xlsr_conformertcm_xlsr_conformertcm_mdt_rawboost3_spoofceleb.txt')}, 'Final_eval': {'protocol': PosixPath('/nvme2/hungdx/code/Lightning-hydra/data/WildSpoof_Final_Eval/Final_eval/protocol.txt'), 'score': PosixPath('/nvme2/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/xlsr_conformertcm_mdt_rawboost3_spoofceleb/Final_eval_wildspoof_Nov3_xlsr_conformertcm_xlsr_conformertcm_mdt_rawboost3_spoofceleb.txt')}, 'lossy_codec': {'protocol': PosixPath('/nvme2/hungdx/code/Lightning-hydra/data/WildSpoof_Final_Eval/lossy_codec/protocol.txt'), 'score': PosixPath('/nvme2/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/xlsr_conformertcm_mdt_rawboost3_spoofceleb/lossy_co

/nvme2/hungdx/code/Lightning-hydra/notebooks/WildSpoof/eval_metrics_DF.py:36: RuntimeWarning: invalid value encountered in divide
  far = np.concatenate((np.atleast_1d(1), nontarget_trial_sums / nontarget_scores.size))  # false acceptance rates
/nvme2/hungdx/code/Lightning-hydra/notebooks/WildSpoof/eval_metrics_DF.py:35: RuntimeWarning: invalid value encountered in divide
  frr = np.concatenate((np.atleast_1d(0), tar_trial_sums / target_scores.size))  # false rejection rates
/nvme2/hungdx/code/Lightning-hydra/notebooks/WildSpoof/eval_metrics_DF.py:35: RuntimeWarning: invalid value encountered in divide
  frr = np.concatenate((np.atleast_1d(0), tar_trial_sums / target_scores.size))  # false rejection rates


,eer,threshold,accuracy,samples,missing_samples
dataset,,,,,
Enroll_bonafide,NaN,-1.742720,99.9435,10619,0
Final_eval,16.2541,4.119256,63.3170,102121,0
TTS_SASV_eval_anony,NaN,-5.164338,73.6786,2800,0
lossy_codec,NaN,-5.175250,43.7647,16150,0


{'eer': 15.26110776403145,
 'threshold': 4.0939507484436035,
 'accuracy': 64.09294555395246,
 'min_score': -5.203054428100586,
 'max_score': 4.93297004699707,
 'samples': 131690,
 'bonafide_samples': 21238,
 'spoof_samples': 110452,
 'dataset': 'POOLED',
 'datasets': 4,
 'dataset_counts': {'Enroll_bonafide': 10619,
  'Final_eval': 102121,
  'TTS_SASV_eval_anony': 2800,
  'lossy_codec': 16150}}

In [2]:
fusion_result["fused_scores"]["TTS_SASV_eval_anony"].to_csv("TTS_SASV_eval_anony_post_eval_cm.txt", sep=" ", header=False, index=False)

In [2]:
fusion_result["fused_scores"]["Final_eval"].to_csv("Final_eval_fusion5_new_v2.txt", sep=" ", header=False, index=False)

In [12]:
# Load the fusion result
fusion_result = pd.read_csv("Final_eval_fusion.txt", sep=" ", header=None)

# Add column names
fusion_result.columns = ["path", "spoof_score", "bona_score"]

# Predict labels based on scores
fusion_result['pred'] = np.where(fusion_result['spoof_score'] < fusion_result['bona_score'], 'bonafide', 'spoof')


# Analyze the prediction results 

# How many samples are predicted as spoof?
print(f"Number of spoof predictions: {len(fusion_result[fusion_result['pred'] == 'spoof'])}")

# How many samples are predicted as bonafide?
print(f"Number of bonafide predictions: {len(fusion_result[fusion_result['pred'] == 'bonafide'])}")




Number of spoof predictions: 57874
Number of bonafide predictions: 44247


# Compare label switching results

In [3]:
import pandas as pd
import numpy as np

Final_eval_fusion3_df = pd.read_csv("Final_eval_fusion3.txt", sep=" ", header=None)
Final_eval_fusion3_df.columns = ["path", "spoof_score", "bona_score"]
Final_eval_fusion3_df['pred'] = np.where(Final_eval_fusion3_df['spoof_score'] < Final_eval_fusion3_df['bona_score'], 'bonafide', 'spoof')



Final_eval_fusion3_df

,path,spoof_score,bona_score,pred
0,data_v2.0/UTT_00000010.flac,2.079617,-2.840805,spoof
1,data_v2.0/UTT_00000011.flac,1.013275,-2.045225,spoof
2,data_v2.0/UTT_00000012.flac,-3.672143,3.867908,bonafide
3,data_v2.0/UTT_00000017.flac,2.362933,-3.187455,spoof
4,data_v2.0/UTT_00000025.flac,4.325562,-4.761941,spoof
...,...,...,...,...
102116,data_v2.0/UTT_00999943.flac,4.156038,-4.783780,spoof
102117,data_v2.0/UTT_00999952.flac,3.953610,-4.654654,spoof
102118,data_v2.0/UTT_00999954.flac,-3.708048,3.976892,bonafide
102119,data_v2.0/UTT_00999955.flac,0.344586,-1.419047,spoof


# conf-1

In [4]:
lora_lossy_codec_conf1_df = pd.read_csv("/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/XLSR_ConformerTCM_LoRA_lossy_codec-conf-1/Final_eval_wildspoof_Nov3_xlsr_conformertcm_lora_XLSR_ConformerTCM_LoRA_lossy_codec-conf-1.txt", sep=" ", header=None)
lora_lossy_codec_conf1_df.columns = ["path", "spoof_score", "bona_score"]
lora_lossy_codec_conf1_df['pred'] = np.where(lora_lossy_codec_conf1_df['spoof_score'] < lora_lossy_codec_conf1_df['bona_score'], 'bonafide', 'spoof')


lora_lossy_codec_conf1_df

,path,spoof_score,bona_score,pred
0,data_v2.0/UTT_00000010.flac,-2.862313,3.140245,bonafide
1,data_v2.0/UTT_00000011.flac,-2.050257,1.902593,bonafide
2,data_v2.0/UTT_00000012.flac,-3.379680,4.438233,bonafide
3,data_v2.0/UTT_00000017.flac,-0.190681,-0.289097,spoof
4,data_v2.0/UTT_00000025.flac,4.314076,-4.837170,spoof
...,...,...,...,...
102116,data_v2.0/UTT_00999943.flac,4.707652,-4.663087,spoof
102117,data_v2.0/UTT_00999952.flac,4.263868,-4.401623,spoof
102118,data_v2.0/UTT_00999954.flac,-3.600915,4.505958,bonafide
102119,data_v2.0/UTT_00999955.flac,-1.847288,1.659248,bonafide


In [9]:
# Check ratio of label switching
print(f"Overall label switching ratio: {len(lora_lossy_codec_conf1_df[lora_lossy_codec_conf1_df['pred'] != Final_eval_fusion3_df['pred']]) / len(lora_lossy_codec_conf1_df) * 100}%")


lossy_codec_df = pd.read_csv("/home/hungdx/code/Lightning-hydra/data/WildSpoof_Final_Eval/lossy_codec/protocol.txt", sep=" ", header=None)
lossy_codec_df.columns = ["path", "subset", "label"]

# Drop columns subset and label
lossy_codec_df = lossy_codec_df.drop(columns=["subset", "label"])

print(f"\nNumber of samples in lossy_codec subset: {len(lossy_codec_df)}")






Overall label switching ratio: 13.760147276270306%

Number of samples in lossy_codec subset: 16150


In [11]:
# Simple label switching analysis for lossy_codec subset

# Get lossy_codec paths
lossy_codec_paths = set(lossy_codec_df['path'])

# Filter both dataframes for lossy_codec samples only
baseline_lossy = Final_eval_fusion3_df[Final_eval_fusion3_df['path'].isin(lossy_codec_paths)].copy()
lora_lossy = lora_lossy_codec_conf1_df[lora_lossy_codec_conf1_df['path'].isin(lossy_codec_paths)].copy()

# Create comparison table
comparison_table = pd.DataFrame({
    'path': baseline_lossy['path'].values,
    'baseline_pred': baseline_lossy['pred'].values,
    'lora_pred': lora_lossy['pred'].values
})

# Add is_switch column
comparison_table['is_switch'] = comparison_table['baseline_pred'] != comparison_table['lora_pred']

# Calculate switching ratio
switching_ratio = (comparison_table['is_switch'].sum() / len(comparison_table)) * 100

print(f"Lossy codec subset analysis:")
print(f"Total samples: {len(comparison_table)}")
print(f"Switched labels: {comparison_table['is_switch'].sum()}")
print(f"Switching ratio: {switching_ratio:.2f}%")
print(f"\nBaseline: {(comparison_table['baseline_pred'] == 'spoof').sum()} spoof, {(comparison_table['baseline_pred'] == 'bonafide').sum()} bonafide")
print(f"LoRA: {(comparison_table['lora_pred'] == 'spoof').sum()} spoof, {(comparison_table['lora_pred'] == 'bonafide').sum()} bonafide")

# Display sample
print("\nFirst 10 rows:")
display(comparison_table.head(10))

# Save to CSV
comparison_table.to_csv("lossy_codec_comparison-conf1.csv", index=False)
print(f"\nSaved to: lossy_codec_comparison-conf1.csv")


Lossy codec subset analysis:
Total samples: 16150
Switched labels: 5546
Switching ratio: 34.34%

Baseline: 8721 spoof, 7429 bonafide
LoRA: 3181 spoof, 12969 bonafide

First 10 rows:


,path,baseline_pred,lora_pred,is_switch
0,data_v2.0/UTT_00000010.flac,spoof,bonafide,True
1,data_v2.0/UTT_00000053.flac,spoof,bonafide,True
2,data_v2.0/UTT_00000065.flac,spoof,bonafide,True
3,data_v2.0/UTT_00000156.flac,spoof,bonafide,True
4,data_v2.0/UTT_00000185.flac,bonafide,bonafide,False
5,data_v2.0/UTT_00000235.flac,spoof,spoof,False
6,data_v2.0/UTT_00000239.flac,spoof,spoof,False
7,data_v2.0/UTT_00000264.flac,spoof,bonafide,True
8,data_v2.0/UTT_00000395.flac,spoof,spoof,False
9,data_v2.0/UTT_00000518.flac,bonafide,bonafide,False



Saved to: lossy_codec_comparison-conf1.csv


# conf-2

In [19]:
lora_lossy_codec_conf2_df = pd.read_csv("/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/XLSR_ConformerTCM_LoRA_lossy_codec-conf-2/Final_eval_wildspoof_Nov3_xlsr_conformertcm_lora_XLSR_ConformerTCM_LoRA_lossy_codec-conf-2.txt", sep=" ", header=None)
lora_lossy_codec_conf2_df.columns = ["path", "spoof_score", "bona_score"]
lora_lossy_codec_conf2_df['pred'] = np.where(lora_lossy_codec_conf2_df['spoof_score'] < lora_lossy_codec_conf2_df['bona_score'], 'bonafide', 'spoof')


lora_lossy_codec_conf2_df

,path,spoof_score,bona_score,pred
0,data_v2.0/UTT_00000010.flac,-3.259226,4.139108,bonafide
1,data_v2.0/UTT_00000011.flac,-2.520342,3.604688,bonafide
2,data_v2.0/UTT_00000012.flac,-3.337655,4.614126,bonafide
3,data_v2.0/UTT_00000017.flac,1.782607,-2.256738,spoof
4,data_v2.0/UTT_00000025.flac,4.020592,-4.748700,spoof
...,...,...,...,...
102116,data_v2.0/UTT_00999943.flac,4.884895,-4.937017,spoof
102117,data_v2.0/UTT_00999952.flac,4.719619,-4.855847,spoof
102118,data_v2.0/UTT_00999954.flac,-3.528116,4.724404,bonafide
102119,data_v2.0/UTT_00999955.flac,-0.281588,0.331105,bonafide


In [20]:
# Check ratio of label switching
print(f"Overall label switching ratio: {len(lora_lossy_codec_conf2_df[lora_lossy_codec_conf2_df['pred'] != Final_eval_fusion3_df['pred']]) / len(lora_lossy_codec_conf2_df) * 100}%")


lossy_codec_df = pd.read_csv("/home/hungdx/code/Lightning-hydra/data/WildSpoof_Final_Eval/lossy_codec/protocol.txt", sep=" ", header=None)
lossy_codec_df.columns = ["path", "subset", "label"]

# Drop columns subset and label
lossy_codec_df = lossy_codec_df.drop(columns=["subset", "label"])

print(f"\nNumber of samples in lossy_codec subset: {len(lossy_codec_df)}")






Overall label switching ratio: 12.00830387481517%

Number of samples in lossy_codec subset: 16150


In [21]:
# Simple label switching analysis for lossy_codec subset

# Get lossy_codec paths
lossy_codec_paths = set(lossy_codec_df['path'])

# Filter both dataframes for lossy_codec samples only
baseline_lossy = Final_eval_fusion3_df[Final_eval_fusion3_df['path'].isin(lossy_codec_paths)].copy()
lora_lossy = lora_lossy_codec_conf2_df[lora_lossy_codec_conf2_df['path'].isin(lossy_codec_paths)].copy()

# Create comparison table
comparison_table = pd.DataFrame({
    'path': baseline_lossy['path'].values,
    'baseline_pred': baseline_lossy['pred'].values,
    'lora_pred': lora_lossy['pred'].values
})

# Add is_switch column
comparison_table['is_switch'] = comparison_table['baseline_pred'] != comparison_table['lora_pred']

# Calculate switching ratio
switching_ratio = (comparison_table['is_switch'].sum() / len(comparison_table)) * 100

print(f"Lossy codec subset analysis:")
print(f"Total samples: {len(comparison_table)}")
print(f"Switched labels: {comparison_table['is_switch'].sum()}")
print(f"Switching ratio: {switching_ratio:.2f}%")
print(f"\nBaseline: {(comparison_table['baseline_pred'] == 'spoof').sum()} spoof, {(comparison_table['baseline_pred'] == 'bonafide').sum()} bonafide")
print(f"LoRA: {(comparison_table['lora_pred'] == 'spoof').sum()} spoof, {(comparison_table['lora_pred'] == 'bonafide').sum()} bonafide")

# Display sample
print("\nFirst 10 rows:")
display(comparison_table.head(10))

# Save to CSV
comparison_table.to_csv("lossy_codec_comparison-conf2.csv", index=False)
print(f"\nSaved to: lossy_codec_comparison-conf2.csv")


Lossy codec subset analysis:
Total samples: 16150
Switched labels: 4508
Switching ratio: 27.91%

Baseline: 8721 spoof, 7429 bonafide
LoRA: 4273 spoof, 11877 bonafide

First 10 rows:


,path,baseline_pred,lora_pred,is_switch
0,data_v2.0/UTT_00000010.flac,spoof,bonafide,True
1,data_v2.0/UTT_00000053.flac,spoof,bonafide,True
2,data_v2.0/UTT_00000065.flac,spoof,bonafide,True
3,data_v2.0/UTT_00000156.flac,spoof,spoof,False
4,data_v2.0/UTT_00000185.flac,bonafide,bonafide,False
5,data_v2.0/UTT_00000235.flac,spoof,spoof,False
6,data_v2.0/UTT_00000239.flac,spoof,spoof,False
7,data_v2.0/UTT_00000264.flac,spoof,bonafide,True
8,data_v2.0/UTT_00000395.flac,spoof,spoof,False
9,data_v2.0/UTT_00000518.flac,bonafide,bonafide,False



Saved to: lossy_codec_comparison-conf2.csv


# conf-3

In [22]:
lora_lossy_codec_conf3_df = pd.read_csv("/home/hungdx/code/Lightning-hydra/logs/results/WildSpoof_Final_Eval/XLSR_ConformerTCM_LoRA_lossy_codec-conf-3/Final_eval_wildspoof_Nov3_xlsr_conformertcm_lora_XLSR_ConformerTCM_LoRA_lossy_codec-conf-3.txt", sep=" ", header=None)
lora_lossy_codec_conf3_df.columns = ["path", "spoof_score", "bona_score"]
lora_lossy_codec_conf3_df['pred'] = np.where(lora_lossy_codec_conf3_df['spoof_score'] < lora_lossy_codec_conf3_df['bona_score'], 'bonafide', 'spoof')


lora_lossy_codec_conf3_df

,path,spoof_score,bona_score,pred
0,data_v2.0/UTT_00000010.flac,-2.827143,3.155631,bonafide
1,data_v2.0/UTT_00000011.flac,-2.531006,3.002384,bonafide
2,data_v2.0/UTT_00000012.flac,-3.402783,4.368815,bonafide
3,data_v2.0/UTT_00000017.flac,2.521153,-3.232053,spoof
4,data_v2.0/UTT_00000025.flac,4.096631,-4.862439,spoof
...,...,...,...,...
102116,data_v2.0/UTT_00999943.flac,4.860074,-4.938394,spoof
102117,data_v2.0/UTT_00999952.flac,4.733957,-4.896578,spoof
102118,data_v2.0/UTT_00999954.flac,-3.594063,4.495626,bonafide
102119,data_v2.0/UTT_00999955.flac,-0.316789,0.070790,bonafide


In [23]:
# Check ratio of label switching
print(f"Overall label switching ratio: {len(lora_lossy_codec_conf3_df[lora_lossy_codec_conf3_df['pred'] != Final_eval_fusion3_df['pred']]) / len(lora_lossy_codec_conf3_df) * 100}%")


lossy_codec_df = pd.read_csv("/home/hungdx/code/Lightning-hydra/data/WildSpoof_Final_Eval/lossy_codec/protocol.txt", sep=" ", header=None)
lossy_codec_df.columns = ["path", "subset", "label"]

# Drop columns subset and label
lossy_codec_df = lossy_codec_df.drop(columns=["subset", "label"])

print(f"\nNumber of samples in lossy_codec subset: {len(lossy_codec_df)}")






Overall label switching ratio: 12.043556173558818%

Number of samples in lossy_codec subset: 16150


In [24]:
# Simple label switching analysis for lossy_codec subset

# Get lossy_codec paths
lossy_codec_paths = set(lossy_codec_df['path'])

# Filter both dataframes for lossy_codec samples only
baseline_lossy = Final_eval_fusion3_df[Final_eval_fusion3_df['path'].isin(lossy_codec_paths)].copy()
lora_lossy = lora_lossy_codec_conf3_df[lora_lossy_codec_conf3_df['path'].isin(lossy_codec_paths)].copy()

# Create comparison table
comparison_table = pd.DataFrame({
    'path': baseline_lossy['path'].values,
    'baseline_pred': baseline_lossy['pred'].values,
    'lora_pred': lora_lossy['pred'].values
})

# Add is_switch column
comparison_table['is_switch'] = comparison_table['baseline_pred'] != comparison_table['lora_pred']

# Calculate switching ratio
switching_ratio = (comparison_table['is_switch'].sum() / len(comparison_table)) * 100

print(f"Lossy codec subset analysis:")
print(f"Total samples: {len(comparison_table)}")
print(f"Switched labels: {comparison_table['is_switch'].sum()}")
print(f"Switching ratio: {switching_ratio:.2f}%")
print(f"\nBaseline: {(comparison_table['baseline_pred'] == 'spoof').sum()} spoof, {(comparison_table['baseline_pred'] == 'bonafide').sum()} bonafide")
print(f"LoRA: {(comparison_table['lora_pred'] == 'spoof').sum()} spoof, {(comparison_table['lora_pred'] == 'bonafide').sum()} bonafide")

# Display sample
print("\nFirst 10 rows:")
display(comparison_table.head(10))

# Save to CSV
comparison_table.to_csv("lossy_codec_comparison-conf3.csv", index=False)
print(f"\nSaved to: lossy_codec_comparison-conf3.csv")


Lossy codec subset analysis:
Total samples: 16150
Switched labels: 4203
Switching ratio: 26.02%

Baseline: 8721 spoof, 7429 bonafide
LoRA: 4592 spoof, 11558 bonafide

First 10 rows:


,path,baseline_pred,lora_pred,is_switch
0,data_v2.0/UTT_00000010.flac,spoof,bonafide,True
1,data_v2.0/UTT_00000053.flac,spoof,bonafide,True
2,data_v2.0/UTT_00000065.flac,spoof,bonafide,True
3,data_v2.0/UTT_00000156.flac,spoof,spoof,False
4,data_v2.0/UTT_00000185.flac,bonafide,bonafide,False
5,data_v2.0/UTT_00000235.flac,spoof,spoof,False
6,data_v2.0/UTT_00000239.flac,spoof,spoof,False
7,data_v2.0/UTT_00000264.flac,spoof,spoof,False
8,data_v2.0/UTT_00000395.flac,spoof,spoof,False
9,data_v2.0/UTT_00000518.flac,bonafide,bonafide,False



Saved to: lossy_codec_comparison-conf3.csv


# new fusion